# **Data Centre Digital Twin (DCDT) Simulator: Quick Start Tutorial - Part 1**

This notebook demonstrates how to use the provided Python modules to
simulate a small data hall, control it with either a PID or
reinforcement‑learning(RL) controller, and visualise the results.  It is
intended for working using Google Colab or other Jupyter environments.

**NOTE:** The ***Project Demo Video*** guides through this notebook to get you started, explains how to use this notebook (this notebook uses other modules which are explained more in the ***Project Visual Documentation***) and get this project running!!!

## Modelling and Control Principles

The simulator discretises the data hall into a 2‑D grid and solves a
simplified heat‑conduction problem using a finite difference scheme.
At each timestep the temperature at an internal grid node `T[i,j]` is
updated to the average of its four neighbours.  This is analogous to
Equation 4 in the [Tutorialspoint article](https://www.tutorialspoint.com/article/modelling-two-dimensional-heat-conduction-problem-using-python#:~:text=%E2%96%BD2T%2Bqgk%3D%E2%88%822T%E2%88%82x2%2B%E2%88%822T%E2%88%82y2%2Bqgk%3D0%E2%80%A6%E2%80%A6) on 2‑D heat conduction,
which states that the temperature at an interior point equals one
quarter of the sum of its immediate neighbours.  Heat generated
by server racks is added to the grid and heat removed by cooling
units is subtracted. Cold‑ and hot‑aisle [airflow assumptions](https://www.parkplacetechnologies.com/blog/data-center-cooling-systems-benefits-comparisons/#:~:text=With%20the%20cold%20aisle%2Fhot%20aisle,either%20%E2%80%9Chot%E2%80%9D%20or%20%E2%80%9Ccold%E2%80%9D%20aisles) are based on industry practice where cold air intakes face rack fronts and hot exhaust air is directed back to the CRAC intakes.

Power Usage Effectiveness (PUE) is used to assess data‑centre
energy efficiency.  [PUE](https://en.wikipedia.org/wiki/Power_usage_effectiveness#:~:text=Ratio%20to%20describe%20data) is defined as the ratio of the total facility
energy to the IT equipment energy.  An ideal PUE equals 1.0,
indicating that all energy supplied to the facility is consumed by
the computing equipment.

The baseline control strategy is a simple PID loop that drives the
maximum rack temperature towards a setpoint by adjusting the cooling
unit control signals.  The PID output is computed as:

$$
u(t) = K_p\,e(t) + K_i \int_{0}^{t} e(\tau)\,d\tau + K_d\,\frac{de(t)}{dt}
$$

Where:

- $u(t)$: Control output (cooling signal, e.g., fan speed or CRAC power level)  
- $e(t) = T_{\text{setpoint}} - T_{\text{measured}}$: Temperature error  
- $K_p$: Proportional gain  
- $K_i$: Integral gain  
- $K_d$: Derivative gain  


- **Proportional Term ($K_p e(t)$)**  
  Responds to the current temperature error.

- **Integral Term ($K_i \int e(t)\,dt$)**  
  Accumulates past error to eliminate steady-state offset.

- **Derivative Term ($K_d \frac{de(t)}{dt}$)**  
  Predicts future error trends and improves system stability.

Here `e(t)` is the error between the desired temperature and the
measured maximum temperature.

An alternative strategy uses reinforcement learning (RL) to learn a
control policy that minimises a cost (e.g., high temperatures and
energy usage).  A tabular Q‑learning agent is provided for
experimentation, and students can optionally install
`stable‑baselines3` to train a PPO agent.

## Install Optional Dependencies

If you wish to train an RL agent with modern algorithms (like PPO), install
[`gymnasium`](https://pypi.org/project/gymnasium/),
[`stable-baselines3`](https://pypi.org/project/stable-baselines3/) and
[`shimmy`](https://pypi.org/project/shimmy/) by running the
following cell.  In Colab you should have a GPU available.

In [1]:
# Run the following line in Colab to install additional packages
!pip install -q stable-baselines3 shimmy gymnasium plotly

## Data Hall Construction
Adjust number of rows and columns below as required before running the following cell. Then in the grid Single-Click to add a Server Rack, and Double-Click to add a Cooling Unit.

In [2]:
from grid_editor import create_grid

server_racks = []
cooling_systems = []

## Initialize Data Hall Grid

rows=10                    ## adjust as required
cols=10                    ## adjust as required

server_racks, cooling_systems = create_grid(rows, cols)

Single Click: Server Rack | Double Click: Cooling Unit


## Create the Simulator, Controller and Dashboard

We import the simulator, controllers and dashboard modules and create the data hall as per our selected design. Feel free to modify the parameters(like setpoints, controllers, your custom dashboard, etc.) to run your tests.

In [3]:
from simulator import create_default_hall
from controllers import PidController, RLController, ControlSwitch
from dashboard import Dashboard

# Build the data hall
hall = create_default_hall(server_racks, cooling_systems, rows, cols)

# NOTE: Initial CPU loads are assigned randomly between 0.3 to 0.7 (inclusive)

# Create controllers
pid_controller = PidController(setpoint=30.0)
rl_controller = RLController()

# Wrap both controllers with a switch
controller = ControlSwitch(pid_controller=pid_controller, rl_controller=rl_controller, use_rl=False)

# Create a dashboard
try:
    dash = Dashboard(hall=hall, controller=controller)
    print("Systems Initialized.")
except Exception as e:
    print(f"Dashboard could not be initialised: {e}")

[0.3 0.4 0.3]
Systems Initialized.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Run the Simulation

The next cell runs the simulator for `steps` number of steps at
`dt` second per step.  The PID controller is used by default.  To
switch to the RL controller at runtime you can call
`controller.toggle()` between steps.  The dashboard will update
live if running in a notebook; otherwise textual summaries will be
printed.

In [4]:
from IPython.display import clear_output, display
import time
import random

# Simulation Parameters - Using small dt(keeping below 1.0 is decent) for thermal stability
steps = 200   ## Number of simulation steps
dt_val = 0.05 ## time interval value in seconds

for i in range(steps):
    # Step the physics
    telemetry = hall.step(dt=dt_val)

    # INTERACTIVE NARRATIVE:
    # Switching to RL control halfway to check the shift in efficiency
    if i == int(steps/2):
        controller.toggle()

    # Compute and apply actions
    actions = controller.compute_actions(hall, telemetry)
    for idx, signal in actions.items():
        hall.cooling_units[idx].control_signal = signal

    # Refresh the UI every 2 steps for smoothness in Colab
    if i % 2 == 0:
        clear_output(wait=True)
        fig = dash.update(telemetry)
        fig.show()

        mode_label = "RL (Q-Learning)" if controller.use_rl else "PID (Baseline)"
        print(f"Step {i+1}/{steps} | Mode: {mode_label}")
        print(f"Time {telemetry['time']:.1f}s, PUE={telemetry['pue']:.3f}, MaxTemp={telemetry['room']['max_temp']:.2f}°C")

    time.sleep(1.0) # Dashboard Update "Frame rate" control

Step 199/200 | Mode: RL (Q-Learning)
Time 10.0s, PUE=1.191, MaxTemp=28.33°C


## Log the Simulation Data
This uses the `logger` module to save simulation data for further detailed analysis. The following cell logs only the final timestep data.

**How to log all timesteps?**

Full log can be collected by first initializing `logger = TelemetryLogger()` before the `for` loop, then using `logger.append(telemetry)` inside the `for` loop of the simulation code cell(above) and using `logger.export_csv("full_telemetry_log.csv")` after the `for` loop execution has ended(outside the `for` loop)

**(Try it out later by modifying the previous block!)**


In [5]:
from logger import TelemetryLogger
import pandas as pd

# Export final state
logger = TelemetryLogger()
logger.append(telemetry)
logger.export_csv("final_telemetry_log.csv")

print("Telemetry successfully exported to final_telemetry_log.csv.")
display(pd.read_csv("final_telemetry_log.csv").head())

Telemetry successfully exported to final_telemetry_log.csv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning:

datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).



,time,pue,rack0_id,rack0_position,rack0_cpu_load,rack0_power_draw,rack0_inlet_temp,rack0_exhaust_temp,rack1_id,rack1_position,...,rack2_position,rack2_cpu_load,rack2_power_draw,rack2_inlet_temp,rack2_exhaust_temp,room_average_temp,room_max_temp,room_min_temp,room_humidity,pue.1
0,10.0,1.189466,0,"[2, 2]",0.3,650.0,28.080803,93.080803,1,"[7, 3]",...,"[5, 7]",0.3,650.0,28.095198,93.095198,24.996222,28.341156,18.937858,50.499167,1.189466
